<a href="https://colab.research.google.com/github/Sahana-R-123/Fashion_MNIST/blob/main/Question3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
#Question 3
import wandb
from keras.datasets import fashion_mnist
import numpy as np

In [2]:
wandb.init(project='Fashion-MNIST',name='optimizers')
(x_train,y_train),(x_test,y_test)=fashion_mnist.load_data()

/usr/local/lib/python3.12/dist-packages/notebook/notebookapp.py:191: SyntaxWarning: invalid escape sequence '\/'
  | |_| | '_ \/ _` / _` |  _/ -_)
wandb: (1) Create a W&B account
wandb: (2) Use an existing W&B account
wandb: (3) Don't visualize my results
wandb: Enter your choice:

 2


wandb: You chose 'Use an existing W&B account'
wandb: Logging into https://api.wandb.ai. (Learn how to deploy a W&B server locally: https://wandb.me/wandb-server)
wandb: Create a new API key at: https://wandb.ai/authorize?ref=models
wandb: Store your API key securely and do not share it.
wandb: Paste your API key and hit enter:

 ··········


wandb: No netrc file found, creating one.
wandb: Appending key for api.wandb.ai to your netrc file: /root/.netrc
wandb: Currently logged in as: rsahana-23 (rsahana-23-anna-university) to https://api.wandb.ai. Use `wandb login --relogin` to force relogin


29515/29515 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
26421880/26421880 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
5148/5148 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step
4422102/4422102 ━━━━━━━━━━━━━━━━━━━━ 0s 0us/step


In [3]:
#preprocess
np.random.seed(42)
x_train = x_train / 255.0
x_test = x_test / 255.0
x_train = x_train.reshape(x_train.shape[0], 784)
x_test = x_test.reshape(x_test.shape[0], 784)

In [4]:
#one hot encode labels
def one_hot_encode(y, num_classes=10):
    return np.eye(num_classes)[y]

y_train_onehot = one_hot_encode(y_train)
y_test_onehot = one_hot_encode(y_test)


In [5]:
#define class
class feedforwardnn:
  def __init__(self, input_size, hidden_layers, output_size, learning_rate):
    self.learning_rate = learning_rate
    self.layers= [input_size] + hidden_layers + [output_size]
    self.weights = [np.random.randn(self.layers[i], self.layers[i + 1])  * np.sqrt(1/self.layers[i]) for i in range(len(self.layers) - 1)]
    self.biases = [np.zeros((1, self.layers[i + 1])) for i in range(len(self.layers) - 1)]

    # Momentum var
    self.vw = [np.zeros_like(w) for w in self.weights]
    self.vb = [np.zeros_like(b) for b in self.biases]

    # RMSProp var
    self.sw = [np.zeros_like(w) for w in self.weights]
    self.sb = [np.zeros_like(b) for b in self.biases]

    # Adam/Nadam var
    self.m_w = [np.zeros_like(w) for w in self.weights]
    self.m_b = [np.zeros_like(b) for b in self.biases]

    self.v_w = [np.zeros_like(w) for w in self.weights]
    self.v_b = [np.zeros_like(b) for b in self.biases]

    self.t = 0
  #activation fn
  def sigmoid(self,x):
    return 1/(1+np.exp(-x))
  def relu(self, x):
    return np.maximum(0, x)

  #derivative
  def sigmoid_der(self,x):
    return x*(1-x)
  def relu_der(self, x):
      return (x > 0).astype(float)
  #output layer
  def softmax(self,x):
    e_x = np.exp(x - np.max(x, axis=1, keepdims=True))
    return e_x/np.sum(e_x,axis=1,keepdims=True)

  #forward propogation
  def forward_prop(self,x):
    activations=[x]
    for i in range(len(self.weights)-1):
      z=np.dot(activations[i],self.weights[i])+self.biases[i]
      #a=self.sigmoid(z)
      a=self.relu(z)
      activations.append(a)
    z= np.dot(activations[-1], self.weights[-1]) + self.biases[-1] #final weights and bias
    final_output=self.softmax(z)
    activations.append(final_output)
    return activations

  #compute loss
  def compute_loss(self,y_true, y_hat, batch_size):

    l = (-1 * np.sum(np.multiply(y_true, np.log(y_hat + 1e-8)))) / batch_size
    return l

  #backward propogation
  def backward_prop(self, x, y_true, activations):
    m=x.shape[0]
    deltas = [activations[-1] - y_true]
    for i in range(len(self.weights)-2, -1, -1):
      delta = np.dot(deltas[0],self.weights[i+1].T)
      #delta = delta * self.sigmoid_der(activations[i+1])
      delta = delta * self.relu_der(activations[i+1])
      deltas.insert(0, delta)
    dW = []
    dB = []
    for i in range(len(self.weights)):
      dw = np.dot(activations[i].T,deltas[i]) / m
      db = np.sum(deltas[i],axis=0,keepdims=True) / m

      dW.append(dw)
      dB.append(db)
    return dW,dB

  #update weights
  def update_para(self, dw, db,optimizer="sgd", beta=0.9,beta1=0.9,beta2=0.999):
    eps = 1e-8
    self.t += 1
    for i in range(len(self.weights)):
      if optimizer == "sgd":
        self.weights[i] -= (self.learning_rate * dw[i])
        self.biases[i] -= (self.learning_rate * db[i])
      elif optimizer == "momentum":
        self.vw[i] = beta * self.vw[i] + self.learning_rate * dw[i]
        self.vb[i] = beta * self.vb[i] + self.learning_rate * db[i]
        self.weights[i] -= self.vw[i]
        self.biases[i] -= self.vb[i]
      elif optimizer == "nesterov":
        prev_vw = self.vw[i]
        prev_vb = self.vb[i]
        self.vw[i] = beta * self.vw[i] + self.learning_rate * dw[i]
        self.vb[i] = beta * self.vb[i] + self.learning_rate * db[i]
        self.weights[i] -= (beta * prev_vw + (1 + beta) * self.vw[i])
        self.biases[i] -= (beta * prev_vb + (1 + beta) * self.vb[i])
      elif optimizer == "rmsprop":
        self.sw[i] = beta2 * self.sw[i] + (1 - beta2) * (dw[i] ** 2)
        self.sb[i] = beta2 * self.sb[i] + (1 - beta2) * (db[i] ** 2)
        self.weights[i] -= (self.learning_rate / (np.sqrt(self.sw[i]) + 1e-8)) * dw[i]
        self.biases[i] -= (self.learning_rate / (np.sqrt(self.sb[i]) + 1e-8)) * db[i]
      elif optimizer == "adam":
        self.m_w[i] = beta1 * self.m_w[i] + (1 - beta1) * dw[i]
        self.m_b[i] = beta1 * self.m_b[i] + (1 - beta1) * db[i]
        self.v_w[i] = beta2 * self.v_w[i] + (1 - beta2) * np.square(dw[i])
        self.v_b[i] = beta2 * self.v_b[i] + (1 - beta2) * np.square(db[i])
        m_w_corrected = self.m_w[i] / (1 - beta1 ** self.t)
        m_b_corrected = self.m_b[i] / (1 - beta1 ** self.t)
        v_w_corrected = self.v_w[i] / (1 - beta2 ** self.t)
        v_b_corrected = self.v_b[i] / (1 - beta2 ** self.t)
        self.weights[i] -= (self.learning_rate / (np.sqrt(v_w_corrected) + eps)) * m_w_corrected
        self.biases[i] -= (self.learning_rate / (np.sqrt(v_b_corrected) + eps)) * m_b_corrected
      elif optimizer == "nadam":
        self.m_w[i] = beta1 * self.m_w[i] + (1 - beta1) * dw[i]
        self.m_b[i] = beta1 * self.m_b[i] + (1 - beta1) * db[i]
        self.v_w[i] = beta2 * self.v_w[i] + (1 - beta2) * np.square(dw[i])
        self.v_b[i] = beta2 * self.v_b[i] + (1 - beta2) * np.square(db[i])
        v_w_corrected = self.v_w[i] / (1 - beta2 ** self.t)
        v_b_corrected = self.v_b[i] / (1 - beta2 ** self.t)
        m_w_lookahead = (beta1 * self.m_w[i] / (1 - beta1 ** self.t)) + (((1 - beta1) * dw[i]) / (1 - beta1 ** self.t))
        m_b_lookahead = (beta1 * self.m_b[i] / (1 - beta1 ** self.t)) + (((1 - beta1) * db[i]) / (1 - beta1 ** self.t))
        self.weights[i] -= (self.learning_rate / (np.sqrt(v_w_corrected) + eps)) * m_w_lookahead
        self.biases[i] -= (self.learning_rate / (np.sqrt(v_b_corrected) + eps)) * m_b_lookahead

  #calculate accuracy
  def calculate_accuracy(self,y_true, y_pred):
      return np.mean(np.argmax(y_true, axis=1) == np.argmax(y_pred, axis=1))

  #Gradient descent
  def gradient_descent(self, x, y_true, epochs, batch_size,optimizer):
      for epoch in range(epochs):
        indices = np.random.permutation(x.shape[0])
        x = x[indices]
        y_true = y_true[indices]
        for i in range(0, x.shape[0], batch_size):
            x_batch = x[i:i+batch_size]
            y_true_batch = y_true[i:i+batch_size]
            activations = self.forward_prop(x_batch)
            dW, dB = self.backward_prop(x_batch, y_true_batch, activations)
            self.update_para(dW, dB,optimizer)
        activations = self.forward_prop(x)
        final_output = activations[-1]
        loss = self.compute_loss(y_true,final_output, x.shape[0])
        accuracy = self.calculate_accuracy(y_true, final_output)
        print(f"Epoch: {epoch+1}, Loss: {loss}, Accuracy: {accuracy}")
        wandb.log({"epoch": epoch+1,"loss": loss,"accuracy": accuracy})

   #Prediction
  def predict(self, X):

          predictions = np.argmax(self.forward_prop(X)[-1],axis=1)
          return predictions

In [7]:
#hyperparameters
input_size = 784
hidden_layers = [128, 64]
output_size = 10
learning_rate = 0.001
epochs = 50
batch_size = 128

#create model
model = feedforwardnn(input_size,hidden_layers,output_size,learning_rate)

# train model
model.gradient_descent(x_train,y_train_onehot,epochs,batch_size,optimizer='adam')

# test accuracy
test_output = model.forward_prop(x_test)[-1]
test_accuracy = model.calculate_accuracy(y_test_onehot,test_output)

print(f"Test Accuracy : {test_accuracy:.4f}")
wandb.log({"test_accuracy": test_accuracy})
wandb.finish()

Epoch: 1, Loss: 0.4163305469818108, Accuracy: 0.8533166666666666
Epoch: 2, Loss: 0.3665207492944138, Accuracy: 0.8684666666666667
Epoch: 3, Loss: 0.3388671453299621, Accuracy: 0.8767666666666667
Epoch: 4, Loss: 0.29828237309354266, Accuracy: 0.8911333333333333
Epoch: 5, Loss: 0.30341417318722835, Accuracy: 0.8870833333333333
Epoch: 6, Loss: 0.2828654185052869, Accuracy: 0.8946166666666666
Epoch: 7, Loss: 0.25515798792473826, Accuracy: 0.9059833333333334
Epoch: 8, Loss: 0.2692362728309867, Accuracy: 0.89955
Epoch: 9, Loss: 0.25171162470825653, Accuracy: 0.9077666666666667
Epoch: 10, Loss: 0.24522673267576248, Accuracy: 0.9101333333333333
Epoch: 11, Loss: 0.26270615406661696, Accuracy: 0.90085
Epoch: 12, Loss: 0.2261010047900907, Accuracy: 0.9168
Epoch: 13, Loss: 0.22480438022620503, Accuracy: 0.91575
Epoch: 14, Loss: 0.21198905832212672, Accuracy: 0.9214666666666667
Epoch: 15, Loss: 0.22559651489469698, Accuracy: 0.9149833333333334
Epoch: 16, Loss: 0.2060641457977581, Accuracy: 0.922433

accuracy,▁▂▂▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▇▆▆▆▇▆▇▇▇▇█▇██▇▇████
epoch,▁▁▁▁▂▂▂▂▂▂▃▃▃▃▃▄▄▄▄▄▅▅▅▅▅▆▆▆▆▆▆▇▇▇▇▇▇███
loss,█▇▆▅▆▅▅▄▅▄▄▄▄▄▃▃▃▃▃▃▃▂▂▂▂▂▂▂▂▁▁▂▁▁▂▂▁▁▁▁
test_accuracy,▁
accuracy,0.96818
epoch,50
loss,0.08647
test_accuracy,0.8878


I used a similar implementation as Question 2 and further extended the backpropagation framework to support multiple optimization algorithms.I implemented multiple optimization algorithms such as SGD, Momentum, Nesterov Accelerated Gradient, RMSProp, Adam, and Nadam inside the update_para function to update the weights efficiently. I also implemented mini-batch gradient descent to support different batch sizes.